# 10 · Modelo completo do passe (calibrado) e moedas · Cover Shadow Value (CSV)

Fase 8f do CRISP-DM (D40). A probabilidade de o passe chegar passa a contar todos os defesas: os que projetam sombra (cone atrás do tronco, como na v3) e, fora do cone, qualquer defesa (sem orientação). Ajusta-se aos passes rasteiros reais (≤ 35 m, D37) e avaliam-se a v3.1 e a v4 com três moedas (valor de cada linha). Só números.

Versões: `v3` e `v4` (a v4 com fundo, D39) como referência; `v3_1_<moeda>` e `v4c_<moeda>` (v4 completo) com as moedas `xt`, `linhas` e `risco`.

In [1]:
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display

from footy.paths import PROJECTS, output_dir

sys.path.insert(0, str(PROJECTS / "cover_shadow"))
import avaliacao_modelo_passe as am
import calcular_modelo_passe as cmp

pd.options.display.float_format = "{:.4f}".format
pd.options.display.width = 200
SAIDA = output_dir("cover_shadow")
with open(SAIDA / "calibracao_passe.json", encoding="utf-8") as ficheiro:
    CAL = json.load(ficheiro)
JOGOS = am.JOGOS

## 1. Calibração

Resultado observado: `successful` = chegou; `unsuccessful` = não chegou (a SkillCorner não separa interceção de bola fora ou passe mal executado); `offside` fica de fora. Linha usada: a do recetor visado, no último frame alinhado da posse.

In [2]:
print({k: CAL[k] for k in ("n_passes", "passes_por_jogo", "pct_chegaram")})
lit = CAL["parametros_literatura"]
aj, aj5 = CAL["ajuste_jogos_juntos"], CAL["ajuste_jogos_juntos_com_bola"]
linhas = []
for n in lit:
    linhas.append({"parametro": n, "literatura": lit[n],
                   "ajustado (4 parâmetros)": aj["parametros"].get(n, np.nan), "erro-padrão": (aj["erro_padrao"].get(n) or np.nan),
                   "ajustado (com vel. da bola)": aj5["parametros"].get(n, np.nan)})
display(pd.DataFrame(linhas).set_index("parametro"))
print("log-verosimilhança: ajustado (4 parâmetros)", aj["log_verosimilhanca"], "| com velocidade da bola:", aj5["log_verosimilhanca"])
print("parâmetros no limite do intervalo de busca:", "4 parâmetros:", aj["no_limite"], "| com bola:", aj5["no_limite"])

{'n_passes': 1408, 'passes_por_jogo': {'1925299': 724, '1996435': 684}, 'pct_chegaram': 87.9971590909091}


,literatura,ajustado (4 parâmetros),erro-padrão,ajustado (com vel. da bola)
parametro,,,,
tempo_reacao,0.7000,1.2660,0.1049,0.6719
velocidade_defesa,5.0000,5.9247,0.7626,9.0000
incerteza_tempo,0.4500,0.7490,0.0546,0.4548
bloqueio_nulo,1.2000,0.3500,0.0000,0.3701
velocidade_bola,15.0000,NaN,NaN,29.9996


log-verosimilhança: ajustado (4 parâmetros) -515.4200629556069 | com velocidade da bola: -490.4164546036476
parâmetros no limite do intervalo de busca: 4 parâmetros: ['bloqueio_nulo'] | com bola: ['velocidade_defesa', 'velocidade_bola']


### 1.1 Ajuste num jogo, teste no outro

In [3]:
f = CAL["fora_da_amostra"]
display(pd.DataFrame({k: {**v["ajuste"]["parametros"], **{"com bola: " + n: x for n, x in v["ajuste_com_bola"]["parametros"].items() if n == "velocidade_bola"}} for k, v in f.items()}))
linhas = []
for v in f.values():
    for modelo, m in v["metricas"].items():
        linhas.append({"ajuste → teste": f"{v['treino']} → {v['teste']}", "modelo": modelo, **m,
                       **({"p<0,5: passes": v["c3_p_menor_que_0_5"][modelo]["passes"], "p<0,5: % chegaram": v["c3_p_menor_que_0_5"][modelo]["pct_chegaram"]}
                          if modelo in v["c3_p_menor_que_0_5"] else {})})
display(pd.DataFrame(linhas))
print("Os dois jogos juntos, previsões fora da amostra (cada jogo previsto com o ajuste do outro):")
display(pd.DataFrame(CAL["metricas_fora_da_amostra_dos_dois_jogos"]).T)
print("Na amostra (ajuste com os dois jogos):")
display(pd.DataFrame(CAL["metricas_na_amostra"]).T)

,ajuste_1925299_teste_1996435,ajuste_1996435_teste_1925299
tempo_reacao,0.9098,1.8555
velocidade_defesa,4.1642,9.0000
incerteza_tempo,0.6381,0.9301
bloqueio_nulo,0.6036,0.3500
com bola: velocidade_bola,29.9969,29.9991


,ajuste → teste,modelo,n,log_loss,brier,"p<0,5: passes","p<0,5: % chegaram"
0,1925299 → 1996435,p_sombras_literatura,684,0.8178,0.1288,42.0000,66.6667
1,1925299 → 1996435,p_completo_literatura,684,0.4950,0.1274,97.0000,62.8866
2,1925299 → 1996435,p_completo_calibrado,684,0.4440,0.1135,55.0000,63.6364
3,1925299 → 1996435,p_completo_calibrado_com_bola,684,0.4131,0.1080,54.0000,61.1111
4,1925299 → 1996435,taxa_base_do_treino,684,0.3819,0.1112,NaN,NaN
5,1996435 → 1925299,p_sombras_literatura,724,0.6636,0.0978,37.0000,43.2432
6,1996435 → 1925299,p_completo_literatura,724,0.3467,0.0857,79.0000,46.8354
7,1996435 → 1925299,p_completo_calibrado,724,0.3417,0.0827,37.0000,24.3243
8,1996435 → 1925299,p_completo_calibrado_com_bola,724,0.3190,0.0774,34.0000,26.4706
9,1996435 → 1925299,taxa_base_do_treino,724,0.3542,0.1006,NaN,NaN


Os dois jogos juntos, previsões fora da amostra (cada jogo previsto com o ajuste do outro):


,n,log_loss,brier
p_completo_calibrado,1408.0000,0.3914,0.0977
p_completo_calibrado_com_bola,1408.0000,0.3648,0.0923


Na amostra (ajuste com os dois jogos):


,n,log_loss,brier
p_sombras_literatura,1408.0000,0.7385,0.1128
p_completo_literatura,1408.0000,0.4187,0.1059
p_completo_calibrado,1408.0000,0.3661,0.0910
p_completo_calibrado_com_bola,1408.0000,0.3483,0.0873


### 1.2 Curva de calibração (por decis da probabilidade prevista; fora da amostra, os dois jogos)

In [4]:
import calibracao_passe as cp

prev = pd.read_parquet(cmp.PASTA_SAIDA / "calibracao_passe_previsoes.parquet")
for nome, col in (("modelo atual (só sombras, literatura)", "p_sombras_literatura"), ("completo, literatura", "p_completo_literatura"),
                  ("completo calibrado (fora da amostra)", "fora_da_amostra_calibrado"), ("completo calibrado (na amostra)", "p_completo_calibrado")):
    print(nome)
    display(cp.curva_de_calibracao(prev["chegou"].to_numpy(), prev[col].to_numpy()).set_index("decil").T)

modelo atual (só sombras, literatura)


decil,0,1,2,3,4,5,6,7,8,9
n,141.0000,141.0000,141.0000,140.0000,141.0000,141.0000,140.0000,141.0000,141.0000,141.0000
previsto,0.3992,0.9831,0.9999,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
observado,0.5957,0.6879,0.9645,0.9214,0.9504,0.9433,0.9500,0.9362,0.9291,0.9220


completo, literatura


decil,0,1,2,3,4,5,6,7,8,9
n,141.0000,141.0000,141.0000,140.0000,141.0000,141.0000,140.0000,141.0000,141.0000,141.0000
previsto,0.1090,0.5737,0.8191,0.9033,0.9411,0.9627,0.9779,0.9891,0.9965,0.9997
observado,0.5390,0.7305,0.8440,0.9071,0.9574,0.9574,0.9286,0.9645,0.9716,1.0000


completo calibrado (fora da amostra)


decil,0,1,2,3,4,5,6,7,8,9
n,141.0000,141.0000,141.0000,140.0000,141.0000,141.0000,140.0000,141.0000,141.0000,141.0000
previsto,0.3407,0.7369,0.8404,0.8848,0.9121,0.9297,0.9460,0.9632,0.9783,0.9934
observado,0.5816,0.7447,0.8723,0.9214,0.9007,0.9504,0.9500,0.9645,0.9574,0.9574


completo calibrado (na amostra)


decil,0,1,2,3,4,5,6,7,8,9
n,141.0000,141.0000,141.0000,140.0000,141.0000,141.0000,140.0000,141.0000,141.0000,141.0000
previsto,0.3565,0.7387,0.8375,0.8883,0.9176,0.9367,0.9527,0.9664,0.9805,0.9941
observado,0.5106,0.7943,0.8582,0.8929,0.9362,0.9574,0.9500,0.9504,0.9645,0.9858


### 1.3 C3 repetido: % de passes com p < 0,5 que chegaram (mesmos passes do modelo)

In [5]:
display(pd.DataFrame(CAL["c3_p_menor_que_0_5_na_amostra"]).T)
print("calibrado, fora da amostra:", CAL["c3_p_menor_que_0_5_fora_da_amostra"])

,passes,chegaram,pct_chegaram
p_sombras_literatura,79.0000,44.0000,55.6962
p_completo_literatura,176.0000,98.0000,55.6818
p_completo_calibrado,85.0000,34.0000,40.0000
p_completo_calibrado_com_bola,80.0000,32.0000,40.0000


calibrado, fora da amostra: {'passes': 92, 'chegaram': 44, 'pct_chegaram': 47.82608695652174}


## 2. Tabelas das versões

Carregam-se as tabelas de `data/processed/cover_shadow/{v3_1, v4_completo_xt, v4_completo_linhas, v4_completo_risco}`. Na v3.1 `E0` é a soma de `ev(p(∅))` das linhas avaliadas (a ameaça disponível com o fundo) e `v` a ameaça negada; na v3, `E0` é a soma do perigo das linhas avaliadas.

In [6]:
V = {j: am.carregar_versoes(j) for j in JOGOS}
CTX = {j: am.contexto_do_jogo(j) for j in JOGOS}
print("Parâmetros do modelo completo:", {k: getattr(cmp.parametros_calibrados(), k) for k in ("tempo_reacao", "velocidade_defesa", "incerteza_tempo", "bloqueio_nulo", "bloqueio_pleno", "velocidade_bola")})

Parâmetros do modelo completo: {'tempo_reacao': 1.2660303969598292, 'velocidade_defesa': 5.924719280730688, 'incerteza_tempo': 0.7490187250717062, 'bloqueio_nulo': 0.3500001081768753, 'bloqueio_pleno': 0.3, 'velocidade_bola': 15.0}


### 2.1 Moeda `linhas`: distribuição do nº de adversários ultrapassados (linhas da v4 completo)

In [7]:
for j in JOGOS:
    print(j)
    display(am.distribuicao_de_linhas(V[j]["v4c_linhas"]["linhas"]).T)

1925299


ganho,0,1,2,3,4,5,6,7,8,9
linhas,34247.0000,4713.0000,6687.0000,4798.0000,4021.0000,2970.0000,3808.0000,1112.0000,564.0000,231.0000
pct,54.2303,7.4631,10.5889,7.5977,6.3673,4.7030,6.0300,1.7609,0.8931,0.3658


1996435


ganho,0,1,2,3,4,5,6,7,8,9,10
linhas,31509.0000,4824.0000,4759.0000,4369.0000,4085.0000,3280.0000,2795.0000,1148.0000,804.0000,282.0000,11.0000
pct,54.4517,8.3365,8.2242,7.5502,7.0594,5.6683,4.8301,1.9839,1.3894,0.4873,0.0190


## 3. Totais por versão e moeda (por jogo) e % de momentos com D < 0 e com v < 0

In [8]:
T = am.tabela_de_totais(V)
display(T.set_index(["jogo", "versao"]))

momentos         E0         v  pct_negada  pct_v_negativa        I        D  pct_D_negativa
jogo    versao                                                                                                  
1925299 v3               8568   266.0772  124.8168     46.9100          0.0000      NaN      NaN             NaN
        v4               8915    27.7410    9.3776     33.8039         28.5474   8.0062   1.3714         42.4790
        v3_1_xt          8563   114.5970   21.8197     19.0404          0.0000      NaN      NaN             NaN
        v4c_xt           8915    27.3244    6.2872     23.0095         24.2513   3.9037   2.3835         33.8755
        v3_1_linhas      8563 38276.4284 3979.3626     10.3964          0.0000      NaN      NaN             NaN
        v4c_linhas       8915  8456.1598  758.4750      8.9695         22.7930 465.2778 293.1972         31.6769
        v3_1_risco       8563   -69.2277   68.8237    -99.4165          0.0000      NaN      NaN             NaN
        v4c_risco        8915     7.8238    7.5327     96.2795         38.2726   8.3104  -0.7777         72.2154
1996435 v3               7737   239.3464   94.3474     39.4188          0.0000      NaN      NaN             NaN
        v4               8145    25.7960    6.1200     23.7247         30.1903   5.8319   0.2881         44.6041
        v3_1_xt          7736   104.7351   17.0805     16.3083          0.0000      NaN      NaN             NaN
        v4c_xt           8145    25.7402    4.2174     16.3846         28.1768   3.0450   1.1724         37.9128
        v3_1_linhas      7736 43112.8969 5064.2874     11.7466          0.0000      NaN      NaN             NaN
        v4c_linhas       8145  8999.0470  816.4204      9.0723         26.4211 566.0482 250.3722         35.1627
        v3_1_risco       7736  -113.4515   81.1465    -71.5253          0.0000      NaN      NaN             NaN
        v4c_risco        8145     3.7597    6.1293    163.0266         38.1338   9.0646  -2.9353         75.0645

## 4. % negada por fase defensiva e por terço, ao lado da v3 (soma dos dois jogos)

In [9]:
print("Por terço")
display(am.por_zona_ou_fase(V, CTX, "terco"))
print("Por fase defensiva")
display(am.por_zona_ou_fase(V, CTX, "fase"))

Por terço


versao,v3,v4,v3_1_xt,v3_1_linhas,v3_1_risco,v4c_xt,v4c_linhas,v4c_risco
terco,,,,,,,,
attacking_third,51.4936,37.8329,22.4907,14.2929,74.4306,25.8522,11.9226,34.6270
defensive_third,25.3211,10.4402,8.6664,10.0910,-36.8517,5.6402,8.0711,-12.4881
middle_third,33.6043,13.6958,12.1547,10.7668,-60.5874,9.4069,8.4780,295.5232


Por fase defensiva


versao,v3,v4,v3_1_xt,v3_1_linhas,v3_1_risco,v4c_xt,v4c_linhas,v4c_risco
fase,,,,,,,,
NaN,0.0038,-16.1561,0.0689,0.0527,-42.8577,-16.3434,-13.4752,9.4628
chaotic,53.6160,12.1935,13.2234,11.8034,-62.4081,6.9314,9.5001,-44.9566
defending_direct,24.1653,12.3316,8.2960,9.3034,-48.4013,2.9744,16.2146,-24.9209
defending_quick_break,53.1859,35.4044,21.6649,9.1499,60.7813,25.8600,7.8765,34.4780
defending_set_play,54.0530,43.3274,27.4691,15.6597,595.5897,23.9795,7.8486,39.3641
defending_transition,39.2778,33.5714,18.7332,9.1210,-631.8851,26.5047,13.0820,83.6286
high_block,24.9080,10.6812,9.4496,10.5730,-35.3965,6.1015,7.8709,-8.9536
low_block,46.2488,33.3761,20.0527,13.0513,120.1583,23.2393,11.4773,33.7230
medium_block,30.4623,11.4740,9.2971,9.7880,-35.7355,7.0744,7.6370,-24.1797


## 5. Lances C06, C03, D01–D03 e W04

E(∅), E(C), I, D e v por versão (I e D só na v4) e o Shapley por defesa (ameaça negada total). D01–D03 e W04 são os da 4.ª ronda (D39); mantêm-se os frames.

In [10]:
PRES = {j: am.presenca_do_jogo(j) for j in JOGOS}
for nome, (jogo, frame) in am.LANCES.items():
    tot, defesas = am.lance(jogo, frame, V[jogo], PRES[jogo][2])
    print(f"=== {nome} · jogo {jogo} · frame {frame}")
    display(tot)
    display(defesas[(defesas.abs() > 1e-9).any(axis=1)])

=== C06 · jogo 1925299 · frame 12660


,v3,v4,v3_1_xt,v4c_xt,v3_1_linhas,v4c_linhas,v3_1_risco,v4c_risco
E0,0.0344,0.0020,0.0124,0.0023,8.3305,1.5957,-0.0102,0.0008
EC,0.0224,0.0020,0.0123,0.0022,8.2412,1.5133,-0.0106,0.0008
I,NaN,0.0000,NaN,0.0000,NaN,0.0132,NaN,0.0001
D,NaN,-0.0000,NaN,0.0001,NaN,0.0692,NaN,-0.0000
v,0.0120,-0.0000,0.0001,0.0001,0.0892,0.0825,0.0004,0.0000


,v3,v4,v3_1_xt,v4c_xt,v3_1_linhas,v4c_linhas,v3_1_risco,v4c_risco
defesa,,,,,,,,
F. Bérenguer,0.0003,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
K. Jelacic,0.0043,-0.0000,0.0001,0.0001,0.0892,0.0825,0.0004,0.0000
A. Zimarino,0.0074,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


=== C03 · jogo 1996435 · frame 39646


,v3,v4,v3_1_xt,v4c_xt,v3_1_linhas,v4c_linhas,v3_1_risco,v4c_risco
E0,0.0274,0.0013,0.0136,0.0013,11.7439,1.0587,-0.0206,-0.0009
EC,0.0202,0.0007,0.0132,0.0009,11.3900,0.8256,-0.0229,-0.0010
I,NaN,0.0004,NaN,0.0000,NaN,0.0308,NaN,0.0002
D,NaN,0.0002,NaN,0.0003,NaN,0.2024,NaN,-0.0001
v,0.0072,0.0006,0.0005,0.0004,0.3540,0.2331,0.0023,0.0001


,v3,v4,v3_1_xt,v4c_xt,v3_1_linhas,v4c_linhas,v3_1_risco,v4c_risco
defesa,,,,,,,,
P. Klimala,0.0042,0.0006,0.0005,0.0004,0.3540,0.2331,0.0023,0.0001
J. Lolley,0.0030,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


... (8 outputs left out)

,v3,v4,v3_1_xt,v4c_xt,v3_1_linhas,v4c_linhas,v3_1_risco,v4c_risco
defesa,,,,,,,,
A. Ouahim,0.0000,-0.0080,0.0000,-0.0053,0.0000,0.0000,0.0010,-0.0052
Léo Sena,0.0464,-0.0053,0.0033,-0.0033,0.0000,0.0000,0.0042,-0.0034
J. Courtney-Perkins,0.2212,-0.0007,0.0010,-0.0063,0.0000,0.0000,0.0010,-0.0066
A. Cáceres,0.0039,-0.0007,0.0003,-0.0005,0.0000,0.0000,0.0004,-0.0005
C. Hollman,0.0104,-0.0013,0.0002,-0.0008,0.0000,0.0000,0.0003,-0.0009


=== W04 · jogo 1996435 · frame 45252


,v3,v4,v3_1_xt,v4c_xt,v3_1_linhas,v4c_linhas,v3_1_risco,v4c_risco
E0,0.2905,0.0963,0.1782,0.1001,0.0000,0.0000,0.1609,0.0971
EC,0.2540,0.1119,0.1715,0.1163,0.0000,0.0000,0.1507,0.1135
I,NaN,0.0008,NaN,0.0004,NaN,0.0000,NaN,0.0005
D,NaN,-0.0163,NaN,-0.0166,NaN,0.0000,NaN,-0.0170
v,0.0365,-0.0156,0.0067,-0.0162,0.0000,0.0000,0.0103,-0.0164


,v3,v4,v3_1_xt,v4c_xt,v3_1_linhas,v4c_linhas,v3_1_risco,v4c_risco
defesa,,,,,,,,
Léo Sena,0.0000,-0.0000,0.0000,-0.0000,0.0000,0.0000,0.0000,-0.0000
J. Courtney-Perkins,0.0186,-0.0087,0.0001,-0.0086,0.0000,0.0000,0.0002,-0.0086
R. Grant,0.0116,-0.0040,0.0058,-0.0044,0.0000,0.0000,0.0088,-0.0045
C. Hollman,0.0063,-0.0028,0.0008,-0.0033,0.0000,0.0000,0.0012,-0.0033


## 6. ρ de Spearman por jogador dentro da posição, entre versões e moedas (dois jogos juntos)

In [11]:
PJ = {n: pd.concat([am.por_jogador(j, V[j], PRES[j][0], PRES[j][1])[n] for j in JOGOS], ignore_index=True) for n in am.VERSOES}
media, minimo = am.matriz_de_rho(PJ)
print("ρ médio (ponderado pelas posições)")
display(media)
print("ρ mínimo (pior posição)")
display(minimo)

ρ médio (ponderado pelas posições)


,v3,v4,v3_1_xt,v3_1_linhas,v3_1_risco,v4c_xt,v4c_linhas,v4c_risco
v3,1.0000,0.7359,0.8790,0.8224,0.6217,0.6462,0.6903,0.6652
v4,0.7359,1.0000,0.8223,0.6415,0.4958,0.8432,0.5952,0.7377
v3_1_xt,0.8790,0.8223,1.0000,0.7607,0.5030,0.7319,0.6220,0.7160
v3_1_linhas,0.8224,0.6415,0.7607,1.0000,0.7987,0.5579,0.7325,0.6530
v3_1_risco,0.6217,0.4958,0.5030,0.7987,1.0000,0.4585,0.6622,0.6359
v4c_xt,0.6462,0.8432,0.7319,0.5579,0.4585,1.0000,0.4627,0.7641
v4c_linhas,0.6903,0.5952,0.6220,0.7325,0.6622,0.4627,1.0000,0.4645
v4c_risco,0.6652,0.7377,0.7160,0.6530,0.6359,0.7641,0.4645,1.0000


ρ mínimo (pior posição)


,v3,v4,v3_1_xt,v3_1_linhas,v3_1_risco,v4c_xt,v4c_linhas,v4c_risco
v3,1.0000,0.0000,0.4000,0.6667,0.4000,-0.0476,0.3000,0.4000
v4,0.0000,1.0000,-0.2000,0.0000,-0.1905,0.5238,0.0000,0.4833
v3_1_xt,0.4000,-0.2000,1.0000,0.4000,-0.4000,0.0714,0.1000,0.4000
v3_1_linhas,0.6667,0.0000,0.4000,1.0000,0.4000,-0.3095,0.0000,0.2381
v3_1_risco,0.4000,-0.1905,-0.4000,0.4000,1.0000,-0.2619,0.2000,0.3571
v4c_xt,-0.0476,0.5238,0.0714,-0.3095,-0.2619,1.0000,-0.6429,0.3833
v4c_linhas,0.3000,0.0000,0.1000,0.0000,0.2000,-0.6429,1.0000,-0.0952
v4c_risco,0.4000,0.4833,0.4000,0.2381,0.3571,0.3833,-0.0952,1.0000
